# 🧠 OO-Native Model — Training Local (VS Code)

Ce notebook tourne **localement** dans VS Code (pas besoin de Google Drive).

⚠️ **Sans GPU NVIDIA** → utilise CPU uniquement → dry-run OK, full training lent.

Pour le full training avec GPU → utilise [Google Colab](https://colab.research.google.com) avec le fichier `train_colab.ipynb`.

In [1]:
# ── 1. Setup chemin ──────────────────────────────────────────────────
import os, sys

# Chemin vers le dossier oo-model (automatique si le notebook est dans notebooks/)
OO_MODEL_PATH = os.path.abspath(os.path.join(os.path.dirname('__file__'), '..'))
# Ou forcer le chemin absolu:
# OO_MODEL_PATH = r'C:\Users\djibi\OneDrive\Bureau\baremetal\oo-model'

os.chdir(OO_MODEL_PATH)
if 'src' not in sys.path:
    sys.path.insert(0, os.path.join(OO_MODEL_PATH, 'src'))

print('CWD:', os.getcwd())
print('Python:', sys.version)

CWD: c:\Users\djibi\OneDrive\Bureau\baremetal\oo-model
Python: 3.13.3 (tags/v3.13.3:6280bb5, Apr  8 2025, 14:47:33) [MSC v.1943 64 bit (AMD64)]


In [2]:
# ── 2. Vérification environnement ────────────────────────────────────
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA disponible: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('Mode CPU — dry-run OK, full training lent (~heures)')
    print('Pour GPU gratuit: https://colab.research.google.com')

PyTorch: 2.9.1+cpu
CUDA disponible: False
Mode CPU — dry-run OK, full training lent (~heures)
Pour GPU gratuit: https://colab.research.google.com


In [ ]:
# ── 3. Construire le dataset OO ──────────────────────────────────────
import subprocess, json

result = subprocess.run([sys.executable, 'scripts/build_dataset.py'], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print('ERREUR:', result.stderr)

with open('data/processed/train.jsonl', encoding='utf-8') as f:
    rows = [json.loads(l) for l in f]
print(f'Dataset: {len(rows)} samples train')
print('Exemple:', rows[0]['instruction'][:80])

In [ ]:
# ── 4. Dry-run (20 steps, rapide sur CPU) ────────────────────────────
# Valide que l'architecture fonctionne correctement
import subprocess

result = subprocess.run(
    [sys.executable, 'scripts/train_oo_native.py', 'configs/oo_native_v1.json', '--dry-run'],
    capture_output=True, text=True
)
print(result.stdout[-3000:] if len(result.stdout) > 3000 else result.stdout)
if result.returncode != 0:
    print('ERREUR:', result.stderr[-2000:])

In [ ]:
# ── 5. Full training (CPU, lent) ─────────────────────────────────────
# ⚠️ Sur CPU: ~2-4 heures pour 5000 steps
# Sur Colab T4: ~25 minutes
# Décommenter pour lancer:

# result = subprocess.run(
#     [sys.executable, 'scripts/train_oo_native.py', 'configs/oo_native_v1.json'],
#     capture_output=False  # affiche en temps réel
# )

print('Full training commenté — décommenter la cellule pour lancer')
print('Recommandation: utiliser Colab T4 (25min) plutôt que CPU local (2-4h)')

In [ ]:
# ── 6. Charger et tester le checkpoint ───────────────────────────────
import torch, json as _json
from pathlib import Path
sys.path.insert(0, 'src')
from oo_model.oo_native import OONativeModel
from oo_model.oo_tokenizer import OOTokenizer

CKPT = Path('checkpoints/oo-native-v1')

if not (CKPT / 'oo_native_v1.pt').exists():
    print('Pas de checkpoint -- lance le dry-run ou le full training d abord')
else:
    cfg = _json.loads((Path('configs') / 'oo_native_v1.json').read_text(encoding='utf-8'))
    model = OONativeModel.from_config(cfg)
    state = torch.load(CKPT / 'oo_native_v1.pt', map_location='cpu', weights_only=False)
    model.load_state_dict(state['model_state_dict'])
    model.eval()

    tok = OOTokenizer()
    tok.load(str(CKPT / 'tokenizer.json'))

    params = sum(p.numel() for p in model.parameters())
    print(f'Modele charge: {params/1e6:.2f}M params')
    print(f'Tokenizer: {len(tok.vocab)} tokens')
    print(f'Etape: {state["step"]}')